# NeuroSeg-X: Novel Deep Learning Framework for Brain MRI

This notebook implements the **NeuroSeg-X** framework for three simultaneous clinical tasks:
1. **Tumor Detection** (Binary)
2. **Tumor Segmentation** (4-class)
3. **Tumor Grading** (LGG / HGG)

**Project Concept:** Mirrors the high-performance pipeline of the Skin Lesion Project, with automated data splitting, sequential execution, and real-time visualization.

In [ ]:
# -------------------------------------------------------------------
# STEP 1: INSTALLS & IMPORTS
# -------------------------------------------------------------------
!pip install -q albumentations opencv-python-headless tensorboard tqdm torchmetrics scikit-learn

import os
import random
import shutil
import zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from google.colab import drive, files

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, f1_score

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# -------------------------------------------------------------------
# STEP 2: CONFIGURATION
# -------------------------------------------------------------------
config = {
    'image_size': [512, 512],
    'batch_size': 4,
    'num_epochs': 50,
    'lr': 1e-4,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'save_dir': '/content/results',
    'data_dir': '/content/data',
    'seed': 42
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(config['seed'])
os.makedirs(config['save_dir'], exist_ok=True)
print("Config saved.")

In [ ]:
# -------------------------------------------------------------------
# STEP 3: DATA EXTRACTION (Direct execution)
# -------------------------------------------------------------------
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/Colab_Notebooks_Data'
if not os.path.exists(drive_path): drive_path = '/content/drive/MyDrive/Colab_Notebook_Data'

datasets = ['brain_glioma.zip', 'brain_menin.zip', 'brain_tumor.zip']
raw_extract = '/content/raw_data'
os.makedirs(raw_extract, exist_ok=True)

print(f"\ud83d? Extracting datasets from {drive_path}...")
for ds in datasets:
    zp = os.path.join(drive_path, ds)
    if os.path.exists(zp):
        print(f"  - Extracting {ds}...")
        with zipfile.ZipFile(zp, 'r') as z: z.extractall(raw_extract)
    else:
        print(f"  ✗ Warning: {ds} not found!")

print("\u2713 Extraction complete.")

In [ ]:
# -------------------------------------------------------------------
# STEP 4: DATA ORGANIZATION (Skin Lesion Style)
# -------------------------------------------------------------------
print("\ud83d? Organizing data into train/val splits...")

data_root = Path(config['data_dir'])
for split in ['train', 'val']: 
    (data_root / split / 'images').mkdir(parents=True, exist_ok=True)
    (data_root / split / 'masks').mkdir(parents=True, exist_ok=True)

# Collect all images and matching masks
all_samples = []
raw_p = Path(raw_extract)

# Robustly find images
image_exts = ['*.png', '*.jpg', '*.jpeg']
found_images = []
for ext in image_exts: found_images.extend(list(raw_p.rglob(ext)))

print(f"Found {len(found_images)} raw image/mask files.")

for img_p in found_images:
    name = img_p.name.lower()
    if 'mask' in name or 'seg' in name: continue
    
    # Match mask
    m_cand = [img_p.parent / f"{img_p.stem}_mask{img_p.suffix}", img_p.parent / f"{img_p.stem}_seg.png"]
    mask_p = next((m for m in m_cand if m.exists()), None)
    
    if mask_p:
        # Determine labels from path
        label_grading = 1 if 'glioma' in str(img_p).lower() else 0
        all_samples.append({'img': img_p, 'mask': mask_p, 'grading': label_grading})

print(f"\u2713 Successfully paired {len(all_samples)} samples.")

# Split and copy (linear execution)
random.shuffle(all_samples)
split_idx = int(0.8 * len(all_samples))
splits = {'train': all_samples[:split_idx], 'val': all_samples[split_idx:]}

for s_name, s_data in splits.items():
    print(f"  Copying {s_name} files...")
    for sample in tqdm(s_data):
        target_img = data_root / s_name / 'images' / sample['img'].name
        target_mask = data_root / s_name / 'masks' / sample['mask'].name
        # Store labels in filename metadata for easier loading
        target_img = data_root / s_name / 'images' / f"g{sample['grading']}_{sample['img'].name}"
        shutil.copy2(sample['img'], target_img)
        shutil.copy2(sample['mask'], data_root / s_name / 'masks' / f"g{sample['grading']}_{sample['mask'].name}")

print("\u2705 Data organization complete!")

In [ ]:
# -------------------------------------------------------------------
# STEP 5: DATASET & MODELS
# -------------------------------------------------------------------
class NeuroSegDataset(Dataset):
    def __init__(self, split, transforms=None):
        self.img_dir = Path(config['data_dir']) / split / 'images'
        self.mask_dir = Path(config['data_dir']) / split / 'masks'
        self.files = sorted(list(self.img_dir.glob('*')))
        self.transforms = transforms
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img_path = self.files[idx]
        mask_path = self.mask_dir / img_path.name
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path), dtype=np.float32)
        if mask.max() > 1: mask /= 255.0
        grading = int(img_path.name[1])
        if self.transforms: 
            aug = self.transforms(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        return {'image': image, 'mask': mask, 'detection': torch.tensor(1.0), 'grading': torch.tensor(grading)}

def get_tf():
    tr = A.Compose([A.Resize(512, 512), A.HorizontalFlip(), A.Normalize(), ToTensorV2()])
    vl = A.Compose([A.Resize(512, 512), A.Normalize(), ToTensorV2()])
    return tr, vl

class DoubleConv(nn.Module):
    def __init__(self, i, o): 
        super().__init__()
        self.c = nn.Sequential(nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(), nn.Conv2d(o, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU())
    def forward(self, x): return self.c(x)

class FAFRM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c//2, 1), nn.ReLU(), nn.Conv2d(c//2, c, 1), nn.Sigmoid())
        self.sa = nn.Sequential(nn.Conv2d(c, 1, 7, 1, 3), nn.Sigmoid())
    def forward(self, x): 
        x = x * self.ca(x)
        return x * self.sa(x) + x

class NeuroSegX(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1 = nn.Sequential(DoubleConv(3, 64), FAFRM(64))
        self.e2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128), FAFRM(128))
        self.seg = nn.Conv2d(128, 4, 1)
        self.det = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 1), nn.Sigmoid())
        self.grd = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 2))
    def forward(self, x):
        x1 = self.e1(x)
        x2 = self.e2(x1)
        return {'segmentation': self.seg(x2), 'detection': self.det(x2), 'grading': self.grd(x2)}

print("Models ready.")

In [ ]:
# -------------------------------------------------------------------
# STEP 6: TRAINING LOOP (Full execution)
# -------------------------------------------------------------------
tr_tf, vl_tf = get_tf()
tr_loader = DataLoader(NeuroSegDataset('train', tr_tf), 4, True)
vl_loader = DataLoader(NeuroSegDataset('val', vl_tf), 4)

model = NeuroSegX().to(config['device'])
opt = optim.AdamW(model.parameters(), 1e-4)
scaler = GradScaler()
c_seg, c_det, c_grd = nn.CrossEntropyLoss(), nn.BCELoss(), nn.CrossEntropyLoss()

print(f"Starting NeuroSeg-X Training for {config['num_epochs']} epochs...")

for epoch in range(config['num_epochs']):
    model.train()
    tl = 0
    for b in tqdm(tr_loader, desc=f"Epoch {epoch+1}"):
        imgs, masks = b['image'].to(config['device']), b['mask'].to(config['device']).long()
        det_gt, grd_gt = b['detection'].to(config['device']), b['grading'].to(config['device'])
        
        with autocast():
            out = model(imgs)
            loss = 0.5*c_seg(out['segmentation'], masks) + 0.25*c_det(out['detection'], det_gt.unsqueeze(1)) + 0.25*c_grd(out['grading'], grd_gt)
        
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tl += loss.item()
    
    # Visualisation (Real-time)
    model.eval()
    with torch.no_grad():
        test_batch = next(iter(vl_loader))
        test_img = test_batch['image'].to(config['device'])
        test_out = model(test_img)
        
        plt.figure(figsize=(10,5))
        plt.subplot(1,2,1); plt.imshow(test_img[0].cpu().permute(1,2,0)); plt.title("MRI Image")
        plt.subplot(1,2,2); plt.imshow(torch.argmax(test_out['segmentation'][0],0).cpu(), cmap='jet'); plt.title("Seg Prediction")
        plt.show()
        
    print(f"Epoch {epoch+1} Complete. Avg Loss: {tl/len(tr_loader):.4f}")

print("Training Finished. Saving model...")
torch.save(model.state_dict(), '/content/neuroseg_final.pth')
files.download('/content/neuroseg_final.pth')